# 07 — Create Renewables.ninja Query Locations

## Research purpose

Create representative weather-profile query points from the current and future scenario campuses.


## Imports

Load the libraries for spatial calculations and table processing.


In [1]:
import math
from pathlib import Path
import re

import pandas as pd

## Project paths

Locate the project folder and define the paths for the scenario campuses, representative query locations and campus-to-query mapping.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "notebooks").is_dir() or not (PROJECT_ROOT / "data").is_dir():
    raise RuntimeError("Launch Jupyter from the folder containing data/ and notebooks/, or from its notebooks/ directory.")

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_FINAL = PROJECT_ROOT / "data" / "final"

CAMPUS_INPUT_PATH = DATA_FINAL / "hyperscale_campuses_current_plus_future_scenario.csv"
QUERY_LOCATIONS_PATH = DATA_INTERIM / "renewables_ninja_query_locations.csv"
CAMPUS_MAPPING_PATH = DATA_INTERIM / "renewables_ninja_campus_query_mapping.csv"

## Query settings

Set the clustering radius to 100 km, select the 2023–2025 weather years and define the Earth radius used for distance calculations.


In [3]:
CLUSTER_RADIUS_KM = 100.0
WEATHER_YEARS = [2023, 2024, 2025]
EARTH_RADIUS_KM = 6371.0088

print(f"Cluster radius: {CLUSTER_RADIUS_KM:g} km")

Cluster radius: 100 km


## Load and check campuses

Read the scenario-campus table and check its required fields, unique identifiers and available coordinates. Prepare the numeric and grouping columns, then inspect the number of campuses, countries and timezones and the total capacity.


In [4]:
required_columns = [
    "campus_group_id",
    "operator_group",
    "country",
    "country_code",
    "world_region",
    "sub_region",
    "timezone",
    "latitude",
    "longitude",
    "record_layer",
    "status",
    "capacity_mw",
    "it_load_mw",
]

campuses = pd.read_csv(CAMPUS_INPUT_PATH)
missing_columns = [column for column in required_columns if column not in campuses.columns]
if missing_columns:
    raise ValueError(f"Campus input is missing required columns: {missing_columns}")

if campuses["campus_group_id"].isna().any() or campuses["campus_group_id"].duplicated().any():
    raise ValueError("Campus input identifiers must be present and unique")

campuses = campuses.copy()
for column in ["latitude", "longitude", "capacity_mw", "it_load_mw"]:
    campuses[column] = pd.to_numeric(campuses[column], errors="coerce")

missing_coordinates = campuses["latitude"].isna() | campuses["longitude"].isna()
if missing_coordinates.any():
    raise ValueError(f"{missing_coordinates.sum()} campus rows are missing coordinates. Fix coordinates before Renewables.ninja clustering.")

for column in ["country_code", "country", "timezone", "world_region", "sub_region", "operator_group", "record_layer", "status"]:
    campuses[column] = campuses[column].fillna("").astype(str)

campuses["capacity_mw"] = campuses["capacity_mw"].fillna(0.0)
campuses["it_load_mw"] = campuses["it_load_mw"].fillna(0.0)

print(f"Scenario campuses: {len(campuses):,}")
print(f"Countries: {campuses['country_code'].nunique():,}")
print(f"Timezones: {campuses['timezone'].nunique():,}")
print(f"Total scenario capacity: {campuses['capacity_mw'].sum():,.2f} MW")
campuses.head()

Scenario campuses: 202
Countries: 36
Timezones: 43
Total scenario capacity: 24,218.92 MW


,campus_group_id,operator_group,country,country_code,world_region,sub_region,timezone,latitude,longitude,facility_count,...,raw_imputed_capacity_mw,raw_imputed_it_load_mw,facility_ids,future_record_ids,source_campus_group_ids,source_types,coordinate_methods,coordinate_confidence,max_distance_to_representative_m,notes
0,CAMPUS-00107,Amazon,United Arab Emirates,ARE,Middle East and Africa,Western Asia,Asia/Dubai,24.893127,55.109898,1,...,NaN,NaN,ARE-AMAZON-0001,NaN,CAMPUS-00107,osm,osm_geometry_or_centroid,medium,0.000000e+00,Scenario campus aggregated from current and fu...
1,CAMPUS-00161,Amazon,United Arab Emirates,ARE,Middle East and Africa,Western Asia,Asia/Dubai,24.268809,54.463322,1,...,NaN,NaN,ARE-AMAZON-0002,NaN,CAMPUS-00161,osm,osm_geometry_or_centroid,medium,0.000000e+00,Scenario campus aggregated from current and fu...
2,CAMPUS-00162,Amazon,United Arab Emirates,ARE,Middle East and Africa,Western Asia,Asia/Dubai,24.708930,54.741704,1,...,NaN,NaN,ARE-AMAZON-0003,NaN,CAMPUS-00162,osm,osm_geometry_or_centroid,medium,7.177505e-10,Scenario campus aggregated from current and fu...
3,CAMPUS-00068,Amazon,Australia,AUS,Asia Pacific,Australia and New Zealand,Australia/Sydney,-34.036750,150.767560,2,...,NaN,NaN,AUS-AMAZON-0001|AUS-AMAZON-0003,NaN,CAMPUS-00068,osm,osm_geometry_or_centroid,medium,1.895360e+02,Scenario campus aggregated from current and fu...
4,CAMPUS-00094,Amazon,Australia,AUS,Asia Pacific,Australia and New Zealand,Australia/Sydney,-33.794864,150.869664,2,...,NaN,NaN,AUS-AMAZON-0002|AUS-AMAZON-0005,NaN,CAMPUS-00094,osm,osm_geometry_or_centroid,medium,7.286535e+01,Scenario campus aggregated from current and fu...


## Define spatial clustering

Define the distance, centroid and identifier helpers, then build clusters by selecting seeds in descending capacity order with campus identifiers resolving ties. Update each spherical capacity-weighted centroid up to five times and use equal weights when no positive capacity weights are available.


In [5]:
# Calculate great-circle distance in kilometres.
def haversine_km(lat1, lon1, lat2, lon2):
    phi1 = math.radians(float(lat1))
    phi2 = math.radians(float(lat2))
    d_phi = math.radians(float(lat2) - float(lat1))
    d_lambda = math.radians(float(lon2) - float(lon1))
    a = math.sin(d_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(d_lambda / 2) ** 2
    return 2 * EARTH_RADIUS_KM * math.asin(math.sqrt(a))


# Compute a spherical capacity-weighted centroid with an equal-weight fallback.
def weighted_centroid(records):
    weights = [max(float(record.get("capacity_mw", 0.0) or 0.0), 0.0) for record in records]
    if not any(weight > 0 for weight in weights):
        weights = [1.0] * len(records)

    x = y = z = total_weight = 0.0
    for record, weight in zip(records, weights):
        lat = math.radians(float(record["latitude"]))
        lon = math.radians(float(record["longitude"]))
        x += weight * math.cos(lat) * math.cos(lon)
        y += weight * math.cos(lat) * math.sin(lon)
        z += weight * math.sin(lat)
        total_weight += weight

    x /= total_weight
    y /= total_weight
    z /= total_weight
    lon = math.atan2(y, x)
    hyp = math.sqrt(x * x + y * y)
    lat = math.atan2(z, hyp)
    return math.degrees(lat), math.degrees(lon), "capacity_weighted_centroid"


# Convert a grouping label into a stable query-ID token.
def safe_token(value):
    token = re.sub(r"[^A-Za-z0-9]+", "-", str(value).strip()).strip("-").upper()
    return token or "UNKNOWN"


# Create the timezone component of a query identifier.
def timezone_token(value):
    parts = [part for part in str(value).replace(chr(92), "/").split("/") if part]
    return safe_token(parts[-1] if parts else value)


# Build deterministic seed-based clusters with bounded centroid updates.
def greedy_spatial_clusters(records, radius_km):
    remaining = set(range(len(records)))
    ordered_indices = sorted(
        range(len(records)),
        key=lambda idx: (-float(records[idx].get("capacity_mw", 0.0) or 0.0), str(records[idx].get("campus_group_id", ""))),
    )
    clusters = []

    while remaining:
        seed = next(index for index in ordered_indices if index in remaining)
        center_lat = float(records[seed]["latitude"])
        center_lon = float(records[seed]["longitude"])
        members = {index for index in remaining if haversine_km(center_lat, center_lon, records[index]["latitude"], records[index]["longitude"]) <= radius_km}

        for _ in range(5):
            center_lat, center_lon, _ = weighted_centroid([records[index] for index in members])
            updated_members = {
                index
                for index in remaining
                if haversine_km(center_lat, center_lon, records[index]["latitude"], records[index]["longitude"]) <= radius_km
            }
            if updated_members == members:
                break
            members = updated_members

        cluster_records = [records[index] for index in sorted(members, key=lambda idx: str(records[idx].get("campus_group_id", "")))]
        clusters.append(cluster_records)
        remaining -= members

    return clusters

## Build query locations and campus mappings

Apply the clustering separately within each country and IANA timezone and assign a query identifier to each cluster. Create a summary for every query point and a mapping for every campus, retaining capacities, membership and representative distances. Use these points to match regional renewable profiles.


In [6]:
query_rows = []
mapping_rows = []
used_query_ids = set()

sort_columns = ["country_code", "timezone", "capacity_mw", "campus_group_id"]
campuses_sorted = campuses.sort_values(sort_columns, ascending=[True, True, False, True]).reset_index(drop=True)

grouped = campuses_sorted.groupby(["country_code", "timezone"], dropna=False, sort=True)

for (country_code, timezone), group in grouped:
    records = group.to_dict("records")
    clusters = greedy_spatial_clusters(records, CLUSTER_RADIUS_KM)
    parent_capacity_mw = sum(float(record.get("capacity_mw", 0.0) or 0.0) for record in records)
    parent_it_load_mw = sum(float(record.get("it_load_mw", 0.0) or 0.0) for record in records)

    for cluster_index, cluster in enumerate(clusters, start=1):
        rep_lat, rep_lon, centroid_method = weighted_centroid(cluster)
        distances = [haversine_km(rep_lat, rep_lon, record["latitude"], record["longitude"]) for record in cluster]
        max_distance_km = max(distances) if distances else 0.0

        base_id = f"RN-{safe_token(country_code)}-{timezone_token(timezone)}-{cluster_index:02d}"
        query_location_id = base_id
        suffix = 2
        while query_location_id in used_query_ids:
            query_location_id = f"{base_id}-{suffix:02d}"
            suffix += 1
        used_query_ids.add(query_location_id)

        country = next((str(record.get("country", "")) for record in cluster if str(record.get("country", "")).strip()), "")
        world_region = next((str(record.get("world_region", "")) for record in cluster if str(record.get("world_region", "")).strip()), "")
        sub_region = next((str(record.get("sub_region", "")) for record in cluster if str(record.get("sub_region", "")).strip()), "")
        operators = sorted({str(record.get("operator_group", "")) for record in cluster if str(record.get("operator_group", "")).strip()})
        record_layers = sorted({str(record.get("record_layer", "")) for record in cluster if str(record.get("record_layer", "")).strip()})
        statuses = sorted({str(record.get("status", "")) for record in cluster if str(record.get("status", "")).strip()})
        campus_refs = sorted(str(record["campus_group_id"]) for record in cluster)
        cluster_capacity_mw = sum(float(record.get("capacity_mw", 0.0) or 0.0) for record in cluster)
        cluster_it_load_mw = sum(float(record.get("it_load_mw", 0.0) or 0.0) for record in cluster)

        query_method = f"{centroid_method}_within_{CLUSTER_RADIUS_KM:g}km_cluster"
        query_rows.append(
            {
                "query_location_id": query_location_id,
                "country_timezone": f"{country} | {timezone}",
                "country_code": country_code,
                "country": country,
                "world_region": world_region,
                "sub_region": sub_region,
                "timezone": timezone,
                "renewable_query_region": f"{country_code} {timezone} cluster {cluster_index:02d}",
                "representative_lat": rep_lat,
                "representative_lon": rep_lon,
                "campus_count": len(cluster),
                "campus_refs": "|".join(campus_refs),
                "operator_count": len(operators),
                "operators": "|".join(operators),
                "record_layers": "|".join(record_layers),
                "statuses": "|".join(statuses),
                "cluster_capacity_mw": cluster_capacity_mw,
                "cluster_it_load_mw": cluster_it_load_mw,
                "parent_timezone_campus_count": len(records),
                "parent_timezone_capacity_mw": parent_capacity_mw,
                "parent_timezone_it_load_mw": parent_it_load_mw,
                "query_level": "global_timezone_spatial_subcluster",
                "query_method": query_method,
                "cluster_radius_km": CLUSTER_RADIUS_KM,
                "max_distance_to_representative_km": max_distance_km,
                "weather_years": "|".join(str(year) for year in WEATHER_YEARS),
                "notes": "Representative point for Renewables.ninja regional renewable availability matching; not an exact renewable generation site.",
            }
        )

        for record, distance_km in zip(cluster, distances):
            mapping_rows.append(
                {
                    "campus_group_id": record["campus_group_id"],
                    "query_location_id": query_location_id,
                    "distance_to_representative_km": distance_km,
                    "country_code": country_code,
                    "country": country,
                    "timezone": timezone,
                    "operator_group": record.get("operator_group", ""),
                    "record_layer": record.get("record_layer", ""),
                    "status": record.get("status", ""),
                    "capacity_mw": record.get("capacity_mw", 0.0),
                    "it_load_mw": record.get("it_load_mw", 0.0),
                    "latitude": record.get("latitude"),
                    "longitude": record.get("longitude"),
                    "representative_lat": rep_lat,
                    "representative_lon": rep_lon,
                    "cluster_radius_km": CLUSTER_RADIUS_KM,
                    "query_method": query_method,
                }
            )

query_locations = pd.DataFrame(query_rows).sort_values(["country_code", "timezone", "query_location_id"]).reset_index(drop=True)
campus_mapping = pd.DataFrame(mapping_rows).sort_values(["query_location_id", "campus_group_id"]).reset_index(drop=True)

## Validate and save the mappings

Check campus coverage, unique identifiers, query references, coordinate bounds and capacity totals. Save both tables and summarize the number of queries, expected API requests and maximum distance to a representative point.


In [7]:
if len(campus_mapping) != len(campuses):
    raise ValueError(f"Campus mapping row count mismatch: {len(campus_mapping)} mapping rows for {len(campuses)} campuses")
if campus_mapping["campus_group_id"].duplicated().any():
    duplicated = campus_mapping.loc[campus_mapping["campus_group_id"].duplicated(), "campus_group_id"].tolist()
    raise ValueError(f"Campus rows mapped more than once: {duplicated[:10]}")
if query_locations[["representative_lat", "representative_lon"]].isna().any().any():
    raise ValueError("Query locations contain missing coordinates")

if set(campus_mapping["campus_group_id"]) != set(campuses["campus_group_id"]):
    raise ValueError("Campus mappings do not cover the current campus input")
if query_locations["query_location_id"].isna().any() or query_locations["query_location_id"].duplicated().any():
    raise ValueError("Query-location identifiers must be present and unique")
if not set(campus_mapping["query_location_id"]).issubset(set(query_locations["query_location_id"])):
    raise ValueError("A campus mapping references an unknown query location")
if not query_locations["representative_lat"].between(-90, 90).all() or not query_locations["representative_lon"].between(-180, 180).all():
    raise ValueError("Query-location representative coordinates are outside valid bounds")
for mapping_column, query_column in [("capacity_mw", "cluster_capacity_mw"), ("it_load_mw", "cluster_it_load_mw")]:
    expected_total = campuses[mapping_column].sum()
    if not math.isclose(campus_mapping[mapping_column].sum(), expected_total, rel_tol=1e-12, abs_tol=1e-9):
        raise ValueError(f"Mapped {mapping_column} does not reconcile with campus inputs")
    if not math.isclose(query_locations[query_column].sum(), expected_total, rel_tol=1e-12, abs_tol=1e-9):
        raise ValueError(f"Query-location {query_column} does not reconcile with campus inputs")

query_locations.to_csv(QUERY_LOCATIONS_PATH, index=False)
campus_mapping.to_csv(CAMPUS_MAPPING_PATH, index=False)

print(f"Query locations: {len(query_locations):,}")
print(f"Expected annual API requests, PV + wind + weather years: {len(query_locations) * 2 * len(WEATHER_YEARS):,}")
print(f"Mapped campuses: {len(campus_mapping):,}")
print(f"Maximum representative distance: {query_locations['max_distance_to_representative_km'].max():.1f} km")
query_locations.head()

Query locations: 87
Expected annual API requests, PV + wind + weather years: 522
Mapped campuses: 202
Maximum representative distance: 71.8 km


,query_location_id,country_timezone,country_code,country,world_region,sub_region,timezone,renewable_query_region,representative_lat,representative_lon,...,cluster_it_load_mw,parent_timezone_campus_count,parent_timezone_capacity_mw,parent_timezone_it_load_mw,query_level,query_method,cluster_radius_km,max_distance_to_representative_km,weather_years,notes
0,RN-ARE-DUBAI-01,United Arab Emirates | Asia/Dubai,ARE,United Arab Emirates,Middle East and Africa,Western Asia,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,54.776669,...,52.770270,3,60.685811,52.770270,global_timezone_spatial_subcluster,capacity_weighted_centroid_within_100km_cluster,100.0,5.132368e+01,2023|2024|2025,Representative point for Renewables.ninja regi...
1,RN-AUS-MELBOURNE-01,Australia | Australia/Melbourne,AUS,Australia,Asia Pacific,Australia and New Zealand,Australia/Melbourne,AUS Australia/Melbourne cluster 01,-37.780600,144.815310,...,95.133793,4,111.138826,95.133793,global_timezone_spatial_subcluster,capacity_weighted_centroid_within_100km_cluster,100.0,1.179806e+01,2023|2024|2025,Representative point for Renewables.ninja regi...
2,RN-AUS-SYDNEY-01,Australia | Australia/Sydney,AUS,Australia,Asia Pacific,Australia and New Zealand,Australia/Sydney,AUS Australia/Sydney cluster 01,-33.879302,150.819662,...,195.490578,5,226.025812,195.490578,global_timezone_spatial_subcluster,capacity_weighted_centroid_within_100km_cluster,100.0,3.035963e+01,2023|2024|2025,Representative point for Renewables.ninja regi...
3,RN-AUT-VIENNA-01,Austria | Europe/Vienna,AUT,Austria,Europe,Western Europe,Europe/Vienna,AUT Europe/Vienna cluster 01,48.143300,14.463300,...,183.486239,2,206.980826,189.452756,global_timezone_spatial_subcluster,capacity_weighted_centroid_within_100km_cluster,100.0,7.900886e-13,2023|2024|2025,Representative point for Renewables.ninja regi...
4,RN-AUT-VIENNA-02,Austria | Europe/Vienna,AUT,Austria,Europe,Western Europe,Europe/Vienna,AUT Europe/Vienna cluster 02,48.140213,16.461708,...,5.966518,2,206.980826,189.452756,global_timezone_spatial_subcluster,capacity_weighted_centroid_within_100km_cluster,100.0,7.900886e-13,2023|2024|2025,Representative point for Renewables.ninja regi...


## Inspect regional query coverage

Summarize query counts, represented campuses, capacity and maximum distances by region, country and timezone. Display the largest groups to inspect the geographical distribution of query locations.


In [8]:
summary = (
    query_locations
    .groupby(["world_region", "country_code", "timezone"], dropna=False)
    .agg(
        query_locations=("query_location_id", "count"),
        represented_campuses=("campus_count", "sum"),
        capacity_mw=("cluster_capacity_mw", "sum"),
        max_distance_km=("max_distance_to_representative_km", "max"),
    )
    .reset_index()
    .sort_values(["query_locations", "capacity_mw"], ascending=[False, False])
)

summary.head(20)

,world_region,country_code,timezone,query_locations,represented_campuses,capacity_mw,max_distance_km
38,North America,USA,America/Chicago,19,33,5209.336106,6.721795e+01
42,North America,USA,America/New_York,8,49,8141.356844,5.158018e+01
41,North America,USA,America/Los_Angeles,5,18,2825.714141,2.928670e+01
18,Europe,GBR,Europe/London,3,6,346.091807,7.181449e+01
22,Europe,NLD,Europe/Amsterdam,2,5,758.654780,1.540713e+01
24,Europe,SWE,Europe/Stockholm,2,7,579.096661,4.079186e+01
16,Europe,FIN,Europe/Helsinki,2,2,286.009375,8.136291e-13
14,Europe,DNK,Europe/Copenhagen,2,4,260.681675,5.899707e+01
26,Latin America,CHL,America/Santiago,2,6,211.516792,2.291764e+01
11,Europe,AUT,Europe/Vienna,2,2,206.980826,7.900886e-13
